# The Same Agent, Using Microsoft Agent FrameworkIn the first notebook you wrote the agent loop yourself: call the model, check for`tool_calls`, dispatch, append the result, repeat, with a turn cap so it couldn't runaway.Now we build the identical agent — same bug, same three tools — on top of**Microsoft Agent Framework**, and the loop disappears into one method call.The purpose is not to show that the framework is better. It is to let you recognisethat the framework is *doing what you already wrote*, so that when it misbehaves youknow exactly where to look.**What you need:** the same `.env` you set up for notebook 1, with your`OPENROUTER_API_KEY` in it. Nothing new to configure.### A note on Agent FrameworkIt is Microsoft's current agent SDK for .NET and Python, and the declared successorto both Semantic Kernel and AutoGen. It talks to Azure OpenAI, OpenAI, Anthropic andBedrock, and it speaks MCP and A2A. We reach OpenRouter through its OpenAI-compatibleendpoint support.

In [ ]:
%pip install --quiet agent-framework-openai python-dotenv pytestimport jsonimport osimport subprocessimport sysfrom pathlib import Pathfrom typing import Annotatedfrom dotenv import find_dotenv, load_dotenvfrom agent_framework import FunctionInvocationContext, function_middleware, toolfrom agent_framework.openai import OpenAIChatCompletionClient# ":free" is OpenRouter's no-cost, rate-limited endpoint for this model.# Drop the suffix for the paid endpoint if the rate limit gets in your way.MODEL = "nvidia/nemotron-3.5-lightning:free"WORKDIR = Path.cwd()load_dotenv(find_dotenv())# NOTE: OpenAIChatCompletionClient targets the Chat Completions API, which is what# OpenRouter serves. There is also an OpenAIChatClient, which targets OpenAI's# Responses API - that one will NOT work through OpenRouter.client = OpenAIChatCompletionClient(    model=MODEL,    api_key=os.environ["OPENROUTER_API_KEY"],    base_url="https://openrouter.ai/api/v1/",)print("model   :", MODEL)print("workdir :", WORKDIR)

## 1. Does the API work?Same smoke test as before, but built the framework's way: wrap the client in an agentwith no tools at all, and run it.Note the `await`. Agent Framework is async throughout. That works directly in anotebook cell; in a plain `.py` script you would need `asyncio.run(...)`.

In [ ]:
poet = client.as_agent(    name="Poet",    instructions="You write short poems and nothing else.",)response = await poet.run("Write a haiku about debugging code.")print(response.text)print("\nusage:", response.usage_details)

## 2. The taskIdentical to the first notebook. The next two cells write the files the agent works on.> **Re-run the `%%writefile buggy.py` cell to put the bug back** and try again from a> clean start.

In [ ]:
%%writefile buggy.pydef add_reading(reading, log=[]):    """Append a sensor reading to a log and return the log."""    log.append(reading)    return logdef average(readings):    """Return the mean of a list of readings."""    return sum(readings) / len(readings)

In [ ]:
%%writefile test_buggy.pyfrom buggy import add_reading, averagedef test_average():    assert average([2, 4, 6]) == 4def test_logs_are_independent():    first = add_reading(1)    second = add_reading(2)    assert first == [1]    assert second == [2]

In [ ]:
print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 3. The toolsThe function bodies are **exactly the same as in the first notebook**. Two things changed:- the `@tool` decorator, which builds the JSON schema from the signature rather than  making you hand-write it, and- `Annotated[str, "..."]`, which is where the parameter descriptions now live.So the forty lines of `TOOLS = [...]` you copied last time are gone. That is the firstreal thing the framework buys you — and notice what it costs: the schema the modelactually receives is now generated out of sight. `print(edit_file.parameters())` if youwant to see it.There is no `DISPATCH` dictionary any more either. The list of tools you hand to theagent *is* the dispatch table, and it is still the agent's entire set of powers.

In [ ]:
@tooldef read_file(path: Annotated[str, "File to read, e.g. buggy.py"]) -> str:    """Read the full contents of a file in the working directory."""    return (WORKDIR / path).read_text()@tooldef edit_file(    path: Annotated[str, "File to edit, e.g. buggy.py"],    old: Annotated[str, "Exact text to replace. Must appear exactly once, whitespace included."],    new: Annotated[str, "Replacement text."],) -> str:    """Replace an exact snippet of text in a file."""    p = WORKDIR / path    text = p.read_text()    if text.count(old) == 0:        return "ERROR: 'old' not found in the file. Read it again and match it exactly."    if text.count(old) > 1:        return "ERROR: 'old' appears more than once. Include more surrounding context."    p.write_text(text.replace(old, new))    return f"ok, edited {path}"@tooldef run_tests() -> str:    """Run the pytest suite and return its output."""    result = subprocess.run(        [sys.executable, "-m", "pytest", "-q"],        capture_output=True, text=True, timeout=60, cwd=WORKDIR,    )    return (result.stdout + result.stderr)[-2000:] or "(no output)"# The schema the model will see, generated for you:print(json.dumps(edit_file.parameters(), indent=2))

## 4. Getting the transcript backHere is the honest cost of the framework. In your hand-written loop you could seeevery tool call, because *you* wrote the `print` statements between the model's replyand the function call. That code now lives inside Agent Framework, so by default youget a final answer and no idea how it was reached.**Middleware** is how you get back in. A function middleware wraps every toolinvocation: you see the call on the way in and the result on the way out, and`call_next()` is the actual invocation happening in the middle.This is a hook into the same loop you wrote by hand — `context.function.name` and`context.arguments` are the model's request, and `context.result` is what yourfunction returned.

In [ ]:
@function_middlewareasync def log_tool_calls(context: FunctionInvocationContext, call_next):    # TODO 1: print the tool name and its arguments before the call.    #         context.function.name  -> the tool being invoked    #         context.arguments      -> a dict (or a pydantic model with .model_dump())    await call_next()          # this is where your function actually runs    # TODO 2: print context.result, truncated to a few hundred characters    pass

## 5. The agentThis is the cell that replaces your whole loop.Everything you wrote by hand — checking whether the reply contained tool calls, parsingthe JSON arguments, looking the function up, appending a `role="tool"` message,going round again — happens inside `run()`.Three things to supply: instructions, the tools and the middleware.

In [ ]:
SYSTEM = (    "You are a debugging assistant. Use the tools to inspect and fix the code. "    "Always run the tests after making an edit, and keep going until they pass. "    "When they pass, reply with one sentence explaining what the bug was.")TASK = "The tests in test_buggy.py are failing. Find the bug in buggy.py and fix it."# TODO 3: build the agent from the client.#         client.as_agent(name=..., instructions=..., tools=[...], middleware=[...])agent = ...

## 6. Run itOne line. Compare it with the twenty you wrote in the first notebook, then read thetranscript and confirm it took the same shape.

In [ ]:
response = await agent.run(TASK)print("\nFINAL:", response.text)print("usage:", response.usage_details)

## 7. Did it actually change the file?

In [ ]:
print(Path("buggy.py").read_text())# run_tests is now a FunctionTool rather than a plain function, so call pytest directlyprint(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 8. What changed, and what didn't| | Hand-written loop | Agent Framework ||---|---|---|| Tool schemas | ~40 lines of JSON you wrote | generated from the signature by `@tool` || The loop | yours, ~20 lines | inside `run()` || Dispatch table | your `DISPATCH` dict | the `tools=[...]` list || Seeing tool calls | your `print` statements | function middleware || Turn cap | your `max_turns` | a framework default you did not choose || Retries, tracing, MCP, multi-provider | write it yourself | included |**What did not change:** the model still only emits requests, your functions still doall the acting, the tool list is still a permission boundary, and the API is stillstateless with the whole history resent each turn. None of that is abstracted away —it is just further from your cursor.### The question worth sitting withWhere is the turn cap now? You wrote `max_turns=8` deliberately in the first notebook.Here you never specified one, so a default you have not read is deciding when youragent gives up — and how much it can spend before it does.That is the real trade with any agent framework. Not that it does something different,but that it makes decisions on your behalf that you would otherwise have had to makeconsciously. Find that setting in the docs before you use this on anything that matters.### Try this1. Take `run_tests` out of the `tools` list and run again. Same experiment as the first   notebook, one word changed.2. Add `max_invocations=2` to the `@tool` decorator on `edit_file` and see what the agent   does when it runs out of edits.